변경 o

# Baseline

In [57]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'grid_search_utils'

Read dataset files

In [27]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [28]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 18 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   case_id                            300000 non-null  int64  
 1   Hospital                           300000 non-null  int64  
 2   Hospital_type                      300000 non-null  int64  
 3   Hospital_city                      300000 non-null  int64  
 4   Hospital_region                    300000 non-null  int64  
 5   Available_Extra_Rooms_in_Hospital  300000 non-null  int64  
 6   Department                         300000 non-null  object 
 7   Ward_Type                          300000 non-null  object 
 8   Ward_Facility                      300000 non-null  object 
 9   Bed_Grade                          299897 non-null  float64
 10  patientid                          300000 non-null  int64  
 11  City_Code_Patient                  2957

In [29]:
train_df.Class.value_counts()

0    178399
1     95881
2     25720
Name: Class, dtype: int64

Make dataset

In [30]:
train_df = train_df.drop(columns=['case_id', 'Hospital', 'patientid', 'City_Code_Patient', 'Patient_Visitors'], axis=1)
train_df = train_df.dropna()
X_train = train_df.iloc[:, :-1]
y_train = train_df.iloc[:, -1]
X_test = test_df.drop(columns=['case_id', 'Hospital', 'patientid', 'City_Code_Patient', 'Patient_Visitors'], axis=1)


print(len(X_train), len(X_test), len(y_train))

299897 18144 299897


In [31]:
train_df.head()

,Hospital_type,Hospital_city,Hospital_region,Available_Extra_Rooms_in_Hospital,Department,Ward_Type,Ward_Facility,Bed_Grade,Type of Admission,Illness_Severity,Age,Admission_Deposit,Class
0,0,9,1,4,gynecology,Q,B,4.0,Trauma,Minor,0-10,7589.0,0
1,0,6,0,3,gynecology,Q,F,1.0,Trauma,Moderate,61-70,5625.0,1
2,3,13,1,2,anesthesia,R,B,4.0,Trauma,Extreme,51-60,4421.0,0
3,0,7,1,4,radiotherapy,Q,C,1.0,Emergency,Moderate,51-60,4003.0,0
4,2,3,2,3,gynecology,Q,A,2.0,Emergency,Moderate,41-50,3864.0,0


In [32]:
from sklearn.model_selection import train_test_split

X = pd.concat([X_train, X_test], axis=0)
X = pd.get_dummies(X, drop_first=True)

X_train = X.iloc[:len(X_train)]
X_test = X.iloc[len(X_train):]
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=0)

In [33]:
X_train.head()

,Hospital_type,Hospital_city,Hospital_region,Available_Extra_Rooms_in_Hospital,Bed_Grade,Admission_Deposit,Department_anesthesia,Department_gynecology,Department_radiotherapy,Department_surgery,...,Illness_Severity_Moderate,Age_11-20,Age_21-30,Age_31-40,Age_41-50,Age_51-60,Age_61-70,Age_71-80,Age_81-90,Age_91-100
262404,1,2,1,3,2.0,4917.0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
270374,0,9,1,3,3.0,4523.0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
52277,1,11,0,2,3.0,3297.0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
118743,0,7,1,6,1.0,4355.0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
289049,1,11,0,2,4.0,4901.0,0,1,0,0,...,1,0,0,0,0,0,1,0,0,0


In [48]:
from sklearn.model_selection import GridSearchCV

param_grid = { 'n_estimators' : [10],
           'max_depth' : [10, 15, 20],
           'min_samples_leaf' : [2, 4, 8],
           'min_samples_split' : [2, 6, 10]
            }

estimator = RandomForestClassifier()

grid_search = GridSearchCV(estimator=estimator,
                          param_grid=param_grid,
                          cv=5,
                          scoring='f1_weighted',
                          n_jobs=-1)

grid_search.fit(X_train, y_train)
grid_search.best_params_

{'max_depth': 20,
 'min_samples_leaf': 2,
 'min_samples_split': 6,
 'n_estimators': 10}

In [ ]:
{'max_depth': 20,
 'min_samples_leaf': 2,
 'min_samples_split': 6,
 'n_estimators': 10}

In [62]:
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100],
              'penalty': ['l1', 'l2']}


grid_search = GridSearchCV(LogisticRegression(solver='liblinear', max_iter=1000, random_state=1), param_grid, cv=5,
                          n_jobs=-1)
grid_search.fit(X_train, y_train)
grid_search.best_params_

{'C': 0.1, 'penalty': 'l1'}

In [38]:
grid_search.best_params_

{'C': 0.001, 'penalty': 'l2'}

In [58]:
parameters = {
    'var_smoothing': [1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13, 1e-14, 1e-15]
}

gnb_clf = GaussianNB()
clf = GridSearchCV(gnb_clf, parameters, cv=5, n_jobs=-1)
clf.fit(X_train, y_train)
clf.best_params_

{'var_smoothing': 1e-07}

Make model & train

In [45]:
model = RandomForestClassifier(n_estimators=100,
                               bootstrap=True,
                               oob_score=False,
                               max_samples=None,
                               random_state=0, 
                              max_depth=12, 
                              min_samples_leaf=10,
                              min_samples_split=8)
model.fit(X_train, y_train)

val_pred = model.predict(X_val)
print(classification_report(y_val, val_pred))

              precision    recall  f1-score   support

           0       0.64      0.95      0.76     17937
           1       0.50      0.17      0.26      9595
           2       0.50      0.00      0.00      2458

    accuracy                           0.62     29990
   macro avg       0.55      0.37      0.34     29990
weighted avg       0.58      0.62      0.54     29990



In [65]:
c1f1 = LogisticRegression(multi_class='multinomial', random_state=1, C=0.1, penalty='l1', solver='saga', max_iter=1000)
c1f2 = RandomForestClassifier(max_depth = 20,
                            min_samples_leaf = 2,
                            min_samples_split = 6,
                            n_estimators = 10)
c1f3 = GaussianNB(var_smoothing=1e-07)
ec1f1 = VotingClassifier(estimators=[
        ('lr', c1f1), ('rf', c1f2), ('gnb', c1f3)], voting='soft')
ec1f1 = ec1f1.fit(X_train, y_train)

val_pred = ec1f1.predict(X_val)
print(classification_report(y_val, val_pred))

C:\Users\tubs5\anaconda3\lib\site-packages\sklearn\linear_model\_sag.py:352: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


              precision    recall  f1-score   support

           0       0.63      0.94      0.76     17937
           1       0.48      0.16      0.24      9595
           2       0.48      0.00      0.01      2458

    accuracy                           0.61     29990
   macro avg       0.53      0.37      0.34     29990
weighted avg       0.57      0.61      0.53     29990



Make test prediction

In [43]:
test_pred = model.predict(X_test)

Export prediction to file

In [44]:
submission_df = pd.read_csv('sample_submission.csv')
submission_df.Class = test_pred
submission_df.to_csv('submission.csv', index=False)